# 🔵 Interacting with Cassandra using Python

This notebook is a hands-on guide that demonstrates how to interact with **Apache Cassandra** (a **Column Family** or **Wide Column Store** NoSQL database) using Python and the `cassandra-driver` library.

## 🛠️ What is Cassandra?
Originally developed by Facebook, Apache Cassandra was designed to manage **large volumes of distributed data** across multiple servers. It offers high availability with no single points of failure. Unlike relational databases and MongoDB, in Cassandra, data modeling must be strictly driven by the **queries** your application will make (Query-Driven Modeling).

### Conceptual Summary

| Property | Details |
|---|---|
| **Paradigm** | Column Family (Wide Column Store) |
| **Query Language** | CQL — Cassandra Query Language (similar to SQL, but with restrictions) |
| **Storage** | Data distributed across partitions based on Partition Key hash |
| **When to use** | Time series (IoT, logs), high-write-scale applications, distributed geographic data, high availability (99.999%) |
| **When NOT to use** | Complex ad-hoc queries, data with many relationships, applications requiring JOINs, small datasets (< 100GB) |

### Terminology: SQL vs Cassandra

| SQL (Relational) | Cassandra |
|---|---|
| Database | **Keyspace** |
| Table | Table |
| Row | **Partition** (may contain multiple internal rows) |
| Primary Key | **Partition Key + Clustering Key** |
| Secondary Index | Secondary Index (less efficient) |

### Local Connection Details (Docker Compose):
- **Host:** `localhost`
- **Port:** `9042`
- **Authentication:** None (default local configuration)

> **⚠️ Important:** Cassandra can take between **1 and 2 minutes** to fully initialize in Docker. Make sure it is already active before running the notebook.

## 📋 Prerequisites

Before running this notebook, make sure that:

1. **Docker** is installed and running on your machine.
2. The project containers have been started with `make up` or `docker compose up -d`.
3. The `cassandra` container is running **and has finished initializing** (check with `docker compose ps` — the status should be `healthy` or `Up`).

> **💡 Tip:** If the connection fails with `NoHostAvailable`, wait another 1-2 minutes and try again. Cassandra is the slowest database to initialize among the 4 in this project.

## 2. Connecting to the Cluster
In Cassandra, we connect to a **cluster** consisting of one or more nodes. The driver manages connection discovery and load balancing automatically.

> **💡 Key Concept:** Unlike Redis and MongoDB, where we connect to a single server, in Cassandra we connect to a **cluster** of nodes. We pass a list of `contact_points` (initial addresses) and the driver automatically discovers the remaining nodes in the network.

**Expected output:**
```
✅ Conectado ao cluster: 'Test Cluster' | Versão do Cassandra: 5.0.x
```

In [ ]:
from cassandra.cluster import Cluster

try:
    # Define the contact points and port
    # In production, you would list multiple IPs: ['10.0.0.1', '10.0.0.2', '10.0.0.3']
    cluster = Cluster(['localhost'], port=9042)
    
    # Establish the communication session
    # The session is thread-safe and can be reused across the entire application
    session = cluster.connect()
    
    # Execute a simple system query to verify the connection
    resultado = session.execute("SELECT cluster_name, release_version FROM system.local")
    for linha in resultado:
        print(f"✅ Connected to cluster: '{linha.cluster_name}' | Cassandra Version: {linha.release_version}")
        
except Exception as e:
    print(f"❌ Error connecting to Cassandra: {e}")
    print("Tip: Wait a bit longer if the Cassandra container has just started.")

---
## 3. Creating a Keyspace
In Cassandra, a **Keyspace** is equivalent to a database in traditional systems. It defines the physical scope of **data replication** — that is, how many copies of the data will be kept and in which datacenters.

We will use:
- **`SimpleStrategy`:** Replication strategy for a single datacenter (ideal for development).
- **`replication_factor: 1`:** Only one copy of the data (since we have a single local node).

> **💡 In production:** We would use `NetworkTopologyStrategy` with `replication_factor: 3` (3 copies on different nodes), ensuring that even if 2 servers go down, the data remains available.

**Expected output:**
```
🏢 Keyspace 'escola' criado ou já existente.
🎯 Sessão apontando para o Keyspace 'escola'.
```

In [ ]:
# Create Keyspace if not exists
# IF NOT EXISTS prevents error if the keyspace was already created
query_keyspace = """
CREATE KEYSPACE IF NOT EXISTS escola 
WITH replication = {
    'class': 'SimpleStrategy', 
    'replication_factor': 1
};
"""
session.execute(query_keyspace)
print("🏢 Keyspace 'escola' created or already exists.")

# Switch to the created keyspace context
# Equivalent to "USE escola;" in CQL
session.set_keyspace('escola')
print("🎯 Session pointing to Keyspace 'escola'.")

---
## 4. Creating Tables and the Primary Key Structure
In Cassandra, the **primary key** (`PRIMARY KEY`) is the most important data modeling concept. It is divided into two parts:

```
PRIMARY KEY ((partition_key), clustering_key)
              ▲                 ▲
              │                 │
    Defines ON WHICH NODE    Defines ORDERING
    the data is stored       within the node
```

1. **Partition Key:** Defines on which cluster node the physical data will be stored. Cassandra applies a hash function on this field to distribute records evenly across nodes.
2. **Clustering Key:** Defines the **physical ordering** of data within the partition.

In this example, we will create an `estudantes` table where the Primary Key is `((curso), id)`: 
- `curso` is the **Partition Key** → all students from the same course are stored **physically together** on the same node.
- `id` is the **Clustering Key** → within the course, students are ordered by ID.

> **💡 Golden Rule:** In Cassandra, data modeling is driven by the **queries** you intend to make, not by the data structure itself. If you need to search for students by course, then `curso` should be the Partition Key.

**Expected output:**
```
📋 Tabela 'estudantes' criada com sucesso!
```

In [ ]:
# Drop table if exists to reset the tests
session.execute("DROP TABLE IF EXISTS estudantes")

# Create Table with composite primary key
# ((curso)) → Partition Key (double parentheses)
# id → Clustering Key (after the comma)
query_tabela = """
CREATE TABLE estudantes (
    curso text,
    id int,
    nome text,
    email text,
    nota float,
    PRIMARY KEY ((curso), id)
);
"""
session.execute(query_tabela)
print("📋 Table 'estudantes' created successfully!")

---
## 5. CRUD — Create (Inserting Data with Prepared Statements)
In Cassandra, it is an excellent practice to use **Prepared Statements**. They bring two important benefits:

1. **Performance:** Cassandra compiles the query only once and reuses the execution plan for all subsequent calls.
2. **Security:** Prevents CQL injection attacks (equivalent to SQL Injection).

> **💡 Key Concept:** The `?` in the query are placeholders that will be replaced with actual values at execution time. This is different from Python f-strings — the values are sent separately to the database.

**Expected output:**
```
✍️ Estudante 'Felipe Souza' inserido no curso 'Ciência da Computação'.
✍️ Estudante 'Ana Costa' inserido no curso 'Ciência da Computação'.
✍️ Estudante 'Carlos Lima' inserido no curso 'Sistemas de Informação'.
✍️ Estudante 'Beatriz Santos' inserido no curso 'Sistemas de Informação'.
```

In [ ]:
# Prepare the insert query (compiled only once by Cassandra)
# The "?" are placeholders that will be replaced with values at execution
query_inserir = "INSERT INTO estudantes (curso, id, nome, email, nota) VALUES (?, ?, ?, ?, ?)"
statement_preparado = session.prepare(query_inserir)

# Student data (tuples in the same order as query fields)
estudantes = [
    ('Ciência da Computação', 1, 'Felipe Souza', 'felipe@email.com', 8.5),
    ('Ciência da Computação', 2, 'Ana Costa', 'ana.costa@email.com', 9.8),
    ('Sistemas de Informação', 3, 'Carlos Lima', 'carlos@email.com', 7.2),
    ('Sistemas de Informação', 4, 'Beatriz Santos', 'beatriz@email.com', 9.0)
]

# Execute sequential batch insertion
# Each call reuses the prepared statement with different values
for est in estudantes:
    session.execute(statement_preparado, est)
    print(f"✍️ Student '{est[2]}' inserted into course '{est[0]}'.")

---
## 6. CRUD — Read (Collecting and Querying Data)

### ⚠️ Cassandra Golden Rule:
You **should only** query data by passing the **Partition Key** in the `WHERE` clause (in our case, the `curso` field). 

Attempting to query data by columns that are not part of the primary key (like `email`) will cause an **error**, unless you force a full scan with `ALLOW FILTERING` — which is **strongly discouraged in production** because it negatively impacts the performance of the entire cluster.

```
✅ SELECT ... WHERE curso = '...'           → Efficient query (uses Partition Key)
✅ SELECT ... WHERE curso = '...' AND id = 1 → Efficient query (uses PK + CK)
❌ SELECT ... WHERE email = '...'            → Error! (email is not part of the key)
⚠️ SELECT ... WHERE email = '...' ALLOW FILTERING → Works, but inefficient!
```

**Expected output:**
```
📖 Consultando alunos de 'Ciência da Computação':
- ID: 1 | Nome: Felipe Souza | Nota: 8.5
- ID: 2 | Nome: Ana Costa | Nota: 9.80...
--------------------------------------------------
📖 Buscando todos os estudantes cadastrados (Select *):
(lista de todos os estudantes)
--------------------------------------------------
Tentando buscar por email...
⚠️ Erro esperado capturado: ...
```

In [ ]:
# === Efficient Query (Filtering by Partition Key: curso) ===
# This is the CORRECT way to query in Cassandra
print("📖 Querying students from 'Ciência da Computação':")
resultados_cc = session.execute("SELECT id, nome, email, nota FROM estudantes WHERE curso = 'Ciência da Computação'")
for linha in resultados_cc:
    print(f"- ID: {linha.id} | Nome: {linha.nome} | Nota: {linha.nota}")

print("-" * 50)

# === Full Query (SELECT * without WHERE) ===
# Works but returns data from ALL partitions
# On large tables, this can be slow and resource-intensive
print("📖 Fetching all registered students (Select *):")
todos = session.execute("SELECT * FROM estudantes")
for estudante in todos:
    print(f"Curso: {estudante.curso} | ID: {estudante.id} | Nome: {estudante.nome} | Email: {estudante.email}")

print("-" * 50)

# === Blocked Query (Trying to filter by non-indexed field) ===
# This demonstrates that Cassandra BLOCKS inefficient queries by default
try:
    print("Trying to search by email (non-indexed field)...")
    session.execute("SELECT * FROM estudantes WHERE email = 'felipe@email.com'")
except Exception as e:
    print(f"⚠️ Expected error caught: {type(e).__name__}")
    print("Cassandra prevents this search because 'email' is not part of the primary key.")
    
    # Using ALLOW FILTERING as a last resort (NOT recommended in production!)
    print("\n🔄 Running with 'ALLOW FILTERING' (full scan — avoid in production!):")
    resultados_filtro = session.execute("SELECT * FROM estudantes WHERE email = 'felipe@email.com' ALLOW FILTERING")
    for linha in resultados_filtro:
        print(f"- Successfully retrieved: {linha.nome} ({linha.curso})")

---
## 7. CRUD — Update (Updating Data)
In Cassandra, writes work as an **Upsert** (Update + Insert). If you run an `UPDATE` or `INSERT` with the same composite primary key, Cassandra simply overwrites the existing data.

> **💡 Key Concept:** In the `WHERE` clause of `UPDATE`, it is mandatory to specify the **complete primary key** (Partition Key + Clustering Key). It is not possible to update multiple records with a single generic query like in SQL.

**Expected output:**
```
🔄 Registro de Felipe Souza atualizado!
👤 Dados atuais: Nome: Felipe Souza | Email: felipe.novo@email.com | Nota: 9.5
```

In [ ]:
# === Update Felipe's grade and email ===
# The FULL primary key must be specified in the WHERE:
#   - Partition Key: curso = 'Ciência da Computação'
#   - Clustering Key: id = 1
query_update = """
UPDATE estudantes 
SET nota = 9.5, email = 'felipe.novo@email.com' 
WHERE curso = 'Ciência da Computação' AND id = 1
"""
session.execute(query_update)
print("🔄 Felipe Souza's record updated!")

# Validate update by fetching the record
# .one() returns a single row (or None if not found)
registro_atualizado = session.execute(
    "SELECT * FROM estudantes WHERE curso = 'Ciência da Computação' AND id = 1"
).one()
print(f"👤 Current data: Name: {registro_atualizado.nome} | Email: {registro_atualizado.email} | Grade: {registro_atualizado.nota}")

---
## 8. CRUD — Delete (Deleting Records)
To delete, we also need to provide the **Partition Key** (and preferably the **Clustering Key**) to locate the exact row.

> **💡 Curiosity:** In Cassandra, `DELETE` does not remove data from disk immediately. It writes a special marker called a **tombstone**. The data is effectively removed in a later process called **compaction**.

**Expected output:**
```
🗑️ Carlos Lima foi removido.
📖 Alunos de Sistemas de Informação restantes:
- ID: 4 | Nome: Beatriz Santos
```

In [ ]:
# === Delete student Carlos Lima ===
# Specifying the complete primary key: Partition Key + Clustering Key
query_delete = "DELETE FROM estudantes WHERE curso = 'Sistemas de Informação' AND id = 3"
session.execute(query_delete)
print("🗑️ Carlos Lima was removed.")

# Show remaining Sistemas de Informação students
print("📖 Remaining Sistemas de Informação students:")
restantes_si = session.execute("SELECT * FROM estudantes WHERE curso = 'Sistemas de Informação'")
for est in restantes_si:
    print(f"- ID: {est.id} | Nome: {est.nome}")

---
## 9. Closing the Connection
It is good practice to shut down the session and cluster after use to free connections and network resources.

In [ ]:
# Shut down the cluster (closes the session and all connections automatically)
cluster.shutdown()
print("🔌 Connection to Cassandra cluster closed successfully.")

---
## 🏁 Conclusion
Congratulations! You have completed the hands-on exercises with Apache Cassandra:
- ✅ Created a Keyspace and configured local replication factors.
- ✅ Modeled a structured table with a **Composite Primary Key** (Partition Key + Clustering Key).
- ✅ Understood the physical distribution of data based on the Partition Key.
- ✅ Used **Prepared Statements** for secure and performant insertion.
- ✅ Understood the **filtering restrictions** and how Cassandra prioritizes efficient queries through keys.

### 🚀 Next Steps
To continue deepening your knowledge of Cassandra, try:
1. **Secondary Indexes (`CREATE INDEX`):** Allow searches by fields outside the primary key (with a performance cost).
2. **Materialized Views:** Create optimized views for different query patterns.
3. **Batch Statements:** Group multiple writes into a single atomic operation.
4. **Consistency Levels:** Control how many nodes must acknowledge a write before returning success (`ONE`, `QUORUM`, `ALL`).
5. **TTL Data:** Set automatic expiration of records with `USING TTL`.

### 📚 Useful References
- [Official Apache Cassandra Documentation](https://cassandra.apache.org/doc/latest/)
- [CQL Reference](https://cassandra.apache.org/doc/latest/cassandra/cql/)
- [DataStax Python Driver](https://docs.datastax.com/en/developer/python-driver/latest/)
- [Data Modeling in Cassandra (DataStax Academy)](https://www.datastax.com/learn/data-modeling-by-example)